In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

sizes = ["pbmc3k", "pbmc6k", "pbmc10k"]
cell_counts = np.array([3000, 6000, 10000])

total_runtime = pd.Series(
    [26.516, 35.551, 73.237],
    index=sizes,
    name="Total instrumented runtime"
)

timing = pd.DataFrame({

    "imports": [3.790, 2.748, 2.459],

    "scanpy settings": [7.456, 7.268, 7.132],

    "argument parsing": [0.001, 0.001, 0.001],

    "I/O": [0.158, 0.192, 0.672],

    "filtering": [0.059, 0.083, 0.424],

    "normalization": [0.058, 0.086, 0.465],

    "highly variable genes": [0.064, 0.081, 0.457],

    "scale": [0.042, 0.076, 0.172],

    "PCA": [0.351, 0.572, 1.076],

    "neighbors": [4.191, 4.231, 30.539],

    "louvain clustering": [0.143, 0.386, 0.781],

    "UMAP": [8.291, 16.079, 14.182],

    "write h5ad": [0.156, 0.210, 0.552],

    "rank_genes_groups": [1.758, 3.539, 14.325],

}, index=sizes)

timing.index.name = "Datasets"

timing#.to_csv("./timing.csv", index=False)

In [ ]:
coarse = pd.DataFrame({
    "Wall time (s)": [27.22, 36.25, 74.30],
    "CPU (%)": [93, 98, 100],
    "Max RSS (KB)": [871992, 954116, 1324180],
}, index=sizes)

coarse.index.name = "Datasets"

coarse

In [ ]:
ax = timing.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7)
)

ax.set_xlabel("Dataset Size", fontsize=13)
ax.set_ylabel("Runtime (seconds)", fontsize=13)
ax.set_title("Instrumented Runtime by Dataset Size", fontsize=15)

plt.xticks(rotation=0)
#plt.xticks([3000, 6000, 10000], ["3K", "6K", "10K"], rotation=0)

plt.legend(
    title="Code Section",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
#sizes = [3000, 6000, 10000]
timing.index = [3000, 6000, 10000]

plt.figure(figsize=(11, 7))

for column in timing.columns:
    plt.plot(
        timing.index,
        timing[column],
        marker="o",
        label=column
    )

plt.xlabel("Dataset Size (Number of Cells)", fontsize=13)
plt.ylabel("Runtime (seconds)", fontsize=13)
plt.title("Runtime Scaling of Scanpy Pipeline Sections", fontsize=15)

plt.xticks([3000, 6000, 10000], ["3K", "6K", "10K"])

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
major_sections = [
    "PCA",
    "neighbors",
    "louvain clustering",
    "UMAP",
    "rank_genes_groups"
]

plt.figure(figsize=(8, 6))

for section in major_sections:
    plt.plot(
        timing.index,
        timing[section],
        marker="o",
        linewidth=2,
        label=section
    )

plt.xlabel("Dataset Size (Number of Cells)", fontsize=13)
plt.ylabel("Runtime (seconds)", fontsize=13)
plt.title("Scaling of Major Computational Steps", fontsize=15)

plt.xticks([3000, 6000, 10000], ["3K", "6K", "10K"])
plt.legend()

plt.tight_layout()
plt.show()

# (e) Bottleneck Analysis

In [ ]:
analysis_sections = [
    "I/O",
    "filtering",
    "normalization",
    "highly variable genes",
    "scale",
    "PCA",
    "neighbors",
    "louvain clustering",
    "UMAP",
    "write h5ad",
    "rank_genes_groups"
]

for size in timing.index:
    section = timing.loc[size, analysis_sections].idxmax()
    runtime = timing.loc[size, analysis_sections].max()

    print(f"{size}: {section} = {runtime:.3f} s")

In [ ]:
X = np.array([3000, 6000, 10000]).reshape(-1, 1)

predictions = []

for section in timing.columns:

    y = timing[section].values

    model = LinearRegression()
    model.fit(X, y)

    predicted_20k = model.predict([[20000]])[0]

    predictions.append({
        "section": section,
        "predicted_20k_sec": max(predicted_20k, 0),
    })

pred_20k = pd.DataFrame(predictions)

pred_20k.sort_values("predicted_20k_sec", ascending=False).reset_index()[['section', 'predicted_20k_sec']]

In [ ]:
print("Expected Running Time:", pred_20k['predicted_20k_sec'].sum(), "(s)")

# cProfile Analysis

In [ ]:
import pstats

profile_files = {
    "PBMC3K": "pbmc3k_rank_genes_groups.prof",
    "PBMC6K": "pbmc6k_rank_genes_groups.prof",
    "PBMC10K": "pbmc10k_rank_genes_groups.prof",
}


def profile_to_df(profile_file):

    stats = pstats.Stats(profile_file)
    rows = []
    for func, stat in stats.stats.items():
        cc, nc, tt, ct, callers = stat
        filename, line, funcname = func
        rows.append({
            "function": funcname,
            "file": filename,
            "line": line,
            "primitive_calls": cc,
            "total_calls": nc,
            "tottime": tt,
            "cumtime": ct
        })
    df = pd.DataFrame(rows)
    return df.sort_values("cumtime", ascending=False)

df3 = profile_to_df("pbmc3k_rank_genes_groups.prof")
df6 = profile_to_df("pbmc6k_rank_genes_groups.prof")
df10 = profile_to_df("pbmc10k_rank_genes_groups.prof")

df3

In [ ]:
def plot_cprofile(profile_file, dataset, top_n=12):

    df = profile_to_df(profile_file)

    df = (
        df.sort_values("cumtime", ascending=False)
          .head(top_n)
          .copy()
    )

    labels = (
        df["function"] +
        "\n" +
        df["file"]
    )

    plt.figure(figsize=(9, 6))

    plt.barh(
        labels[::-1],
        df["cumtime"][::-1]
    )

    plt.xlabel("Cumulative Time (seconds)")
    plt.ylabel("Function")
    plt.title(f"{dataset}: cProfile of rank_genes_groups")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_cprofile("pbmc3k_rank_genes_groups.prof", "PBMC3K")

plot_cprofile("pbmc6k_rank_genes_groups.prof", "PBMC6K")

plot_cprofile("pbmc10k_rank_genes_groups.prof", "PBMC10K")